In [0]:

dbutils.widgets.dropdown("RUN_ON_TEST_USERS", "True", ["True", "False"], "Запуск на тестовых игроках:")
dbutils.widgets.combobox(
    name="LEVEL_COHORT_FRACTION", 
    defaultValue="100", 
    choices=["1", "10", "100", "1000"], 
    label="Кратность групп уровней"
)


In [0]:
import pandas as pd
import plotly.express as px
from pyspark.sql import DataFrame, Window as W, functions as F, types as T

In [0]:
def write_partitioned(df, path, partition_col="partition_date"):
  """Перезаписать DataFrame в parquet с партиционированием по дате."""
  (
    df.repartition(partition_col)
      .write.mode("overwrite")
      .partitionBy(partition_col)
      .parquet(path)
  )

# Сборка таблицы для MM

In [0]:
# Адрес индивидуальной директории
USER = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
PATH = f"dbfs:/Users/{USER}/progress_analysis_dashboard/"

# Флаг запуска на тестовых пользователях (меняет и данные, и директорию вывода)
RUN_ON_TEST_USERS = dbutils.widgets.get("RUN_ON_TEST_USERS") == "True"
if RUN_ON_TEST_USERS:
  PATH = PATH + "test_users/"

print("Working directory: " + PATH)

# Таблицы, которые понадобятся
levels = spark.table("bronze.levels_mm_amp")
user_state = spark.table("silver.player_state_mm")
revenue = spark.table("bronze.revenue_mm_amp")  # пока не используется, оставлено для downstream
raw_objects_table = "game_data_prod.analytics_voki.raw_objects_mm"


def write_partitioned(df, path, partition_col="partition_date"):
  """Перезаписать DataFrame в parquet с партиционированием по дате."""
  (
    df.repartition(partition_col)
      .write.mode("overwrite")
      .partitionBy(partition_col)
      .parquet(path)
  )

In [0]:
# ОБЩИЕ КОНСТАНТЫ ДЛЯ ПАЙПЛАЙНА

# Цвета для исходов
COLOR_MAP = {
    "close_fail": "#1f77b4",
    "far_fail":   "#d62728",
    "close_win":  "#ffcc00",
    "far_win":    "#2ca02c",
    "unknown":    "#7f7f7f",
}

# Флаги исходов match3: короткое имя -> значение reason_seg
OUTCOME_FLAGS = {
    "FW": "far_win",
    "CW": "close_win",
    "CF": "close_fail",
    "FF": "far_fail",
}
# Пары исходов для расчёта долей "внутри выигрышей / внутри проигрышей"
WIN_PAIR = ["FW", "CW"]
FAIL_PAIR = ["FF", "CF"]

# Ключи, по которым собираем объекты
KEY_USER = ["client_time", "balance_id", "user_id", "partition_date"]
KEY_MAP = ["partition_date", "level_cohort"]

# Гранулярность, по которой строятся когорты уровней
LEVEL_COHORT_FRACTION = float(dbutils.widgets.get("LEVEL_COHORT_FRACTION"))

# Минимальная точка данных, на которые вообще смотрим (грубый фильтр по времени)
DATE_HARD_CUTOFF = "2026-01-01"

# Тестовые пользователи
TEST_USERS = [
    "17042026-082657-YpDIa7ZW", "18042026-082019-s5iwH0sC", "18032026-085553-ouCsFujZ",
    "17032026-024147-cRN1sFCc", "19032026-012513-ZRvSwYvq", "19032026-015621-0PNwoirg",
    "19032026-132942-XYcsux6B", "21032026-103441-GgG2vP4L", "22032026-131537-80SeC6aq",
    "22032026-140442-1vyKygOp",
]

In [0]:
import json
unique_ab_groups_mm = set()
for row in ab:
  ab_group = row.ab_group
  if ab_group:
    ab_group = json.loads(ab_group)
    for g in ab_group:
      unique_ab_groups_mm.add(g)

unique_ab_groups_mm = list(unique_ab_groups)
unique_ab_groups_mm

In [0]:
# по наличию ДРШ
# по ЛД-шным градациям сложности уровня
# в чем смысл: по количеству этого показателя сможем смотреть, нет ли подозрительного дропа в какой-нибудь из дней (не отключилась ли по ошибке АП на некоторых уровнях), то есть тут важны не столько попытки, сколько уровни, но если именно уровни с АП выводить сложно, то тут обсуждаемо.

In [0]:
# Пользователи, которых берём в анализ
if RUN_ON_TEST_USERS:
  user_filter = F.col("user_id").isin(TEST_USERS)
else:
  user_filter = F.col("cheater").isin("Fair", "Soft")

relevant_users = (
  user_state
  .filter(user_filter)
  .select("user_id", "traffic_type")
  .distinct()
)

# Забираем нужные поля и приводим к гранулярности "игрок-попытка-момент времени"
objects = (
  levels
  .filter(F.col("event_type") == "m3.level_finished")
  .filter(F.col("reason").isin("completed", "failed"))
  .filter(F.col("chain").isNull())
  .filter(F.col("client_time") > DATE_HARD_CUTOFF)
  .filter(F.col("partition_date") > DATE_HARD_CUTOFF)
  # Оставляем только релевантных пользователей
  .join(F.broadcast(relevant_users), on="user_id", how="inner")
  # Дополнительные поля для анализа
  .withColumn("failed", F.when(F.col("reason") == "failed", 1).otherwise(0))
  .withColumn(
    "balance_id",
    F.expr("CAST(regexp_extract(get_json_object(event_payload, '$.balance_id'), 'st0*([0-9]+)', 1) AS INT)"),
  )
  .select(
    *KEY_USER,
    "traffic_type",
    "payer_type",
    "failed",
    "ab_group",
    F.json_tuple("event_payload", "reason_seg", "attempt").alias("reason_seg", "attempt"),
    F.json_tuple("user_payload", "tech.platform_name").alias("platform_name"),
  )
  .fillna("unknown", subset="reason_seg")
  .withColumn("first_attempt", F.when(F.col("attempt") == 1, 1).otherwise(0))
  .withColumn("level_cohort", F.floor(F.col("balance_id") / F.lit(LEVEL_COHORT_FRACTION)))
)

# Разворачиваем исходы match3 в бинарные флаги одним циклом
for flag, seg in OUTCOME_FLAGS.items():
  objects = objects.withColumn(flag, F.when(F.col("reason_seg") == seg, 1).otherwise(0))

# Пишем raw objects в Unity Catalog table, которую читает Databricks App
(
  objects.write
  .mode("overwrite")
  .option("mergeSchema", "true")
  .partitionBy("partition_date")
  .saveAsTable(raw_objects_table)
)

# Графики строим только на ограниченном (тестовом) датасете
if RUN_ON_TEST_USERS:
  df = spark.table(raw_objects_table).toPandas()
  display(df)
  print(f"Размер датасета: {len(df)}")
  for user in TEST_USERS:
    display(px.scatter(df[df.user_id == user], x="client_time", y="level_cohort", color="reason_seg"))

# Сборка таблиц для MyM

In [0]:
# Адрес индивидуальной директории (тот же дашборд, отдельная поддиректория под MyM)
USER = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
PATH = f"dbfs:/Users/{USER}/progress_analysis_dashboard/mym/"

# Флаг запуска на тестовых пользователях (меняет и данные, и директорию вывода)
RUN_ON_TEST_USERS = False
if RUN_ON_TEST_USERS:
  PATH = PATH + "test_users/"

print("Working directory: " + PATH)

# reduce нужен, чтобы склеить годовые таблицы исходов в одну
from functools import reduce

# Таблицы, которые понадобятся (MyM)
levels = spark.table("bronze.levels_mym_amp")
user_state = spark.table("silver.player_state_mym")
revenue = spark.table("bronze.revenue_mym_amp")  # пока не используется, оставлено для downstream

# В MyM исходы match3 лежат НЕ в event_payload (как в MM), а в отдельных годовых
# таблицах с уже готовыми бинарными колонками FW/CW/CF/FF. Прицепим их джойном ниже.
OUTCOMES_TABLE = "game_data_prod.analytics_voki.dk_mym_outcomes_ML_churn_2026"

# Дурацкий костыль из-за того что мне нехочется идти просить еще доступов на таблицы новые
raw_objects_table = "game_data_prod.analytics_voki.raw_objects_mm_test_users"


In [0]:
# ОБЩИЕ КОНСТАНТЫ ДЛЯ ПАЙПЛАЙНА (MyM)

# Цвета для исходов
COLOR_MAP = {
    "close_fail": "#1f77b4",
    "far_fail":   "#d62728",
    "close_win":  "#ffcc00",
    "far_win":    "#2ca02c",
    "unknown":    "#7f7f7f",
}

# В MyM исходы приезжают отдельной таблицей уже готовыми колонками FW/CW/CF/FF,
# поэтому маппинг reason_seg -> флаг (как в MM) здесь не нужен — достаточно списка имён.
OUTCOME_FLAGS = ["FW", "CW", "CF", "FF"]
# Пары исходов для расчёта долей "внутри выигрышей / внутри проигрышей"
WIN_PAIR = ["FW", "CW"]
FAIL_PAIR = ["FF", "CF"]

# Ключи, по которым собираем объекты
KEY_USER = ["client_time", "balance_id", "user_id", "partition_date"]
KEY_MAP = ["partition_date", "level_cohort"]

# Гранулярность, по которой строятся когорты уровней
LEVEL_COHORT_FRACTION = 100

# Корректировка дрифта client_time: выкидываем объекты со временем, "уехавшим" в будущее
MAX_FUTURE_DRIFT = 3 * 86400  # client_time - event_time <= 3 дня

# Минимальная точка данных, на которые вообще смотрим (грубый фильтр по времени) — как в MM
DATE_HARD_CUTOFF = "2026-01-01"

# Тестовые пользователи
TEST_USERS = [
    "12042026-195125-vxVRqdYZ", "16042026-121733-FZRhGlZB", "08042026-115122-22VS1Pg2",
    "29032026-212527-hBDzzfIP", "23092023-194259-tt60IN4o", "16042026-145502-NrExEToN",
    "20112023-154120-bFIT5fkI",
]

In [0]:
import json
unique_ab_groups_mym = set()
for row in ab:
  ab_group = row.ab_group
  if ab_group:
    ab_group = json.loads(ab_group)
    for g in ab_group:
      unique_ab_groups_mym.add(g)

unique_ab_groups_mym = list(unique_ab_groups)
unique_ab_groups_mym

In [0]:
# Пользователи, которых берём в анализ (как в MM: честные + мягкие читеры, без ограничения по payer_type)
if RUN_ON_TEST_USERS:
  user_filter = F.col("user_id").isin(TEST_USERS)
else:
  user_filter = F.col("cheater").isin("Fair", "Soft")

relevant_users = (
  user_state
  .filter(user_filter)
  .select("user_id", "traffic_type")
  .distinct()
)

# Исходы match3: в MyM они лежат в отдельных годовых таблицах с уже готовыми бинарными
# колонками FW/CW/CF/FF. Склеиваем годы в одну таблицу и оставляем ключ + флаги.
outcomes = spark.table(OUTCOMES_TABLE)
# Джойнимся по тем ключам из KEY_USER, что реально присутствуют в таблице исходов
outcome_keys = [c for c in KEY_USER if c in outcomes.columns]
outcomes = outcomes.select(*outcome_keys, *OUTCOME_FLAGS)

# Забираем нужные поля и приводим к гранулярности "игрок-попытка-момент времени"
objects = (
  levels
  # Корректировка дрифта client_time: выкидываем объекты со временем, "уехавшим" в будущее
  .withColumn("drift_sec", F.col("client_time").cast("long") - F.col("event_time").cast("long"))
  .filter(F.col("client_time").isNotNull() & F.col("event_time").isNotNull())
  .filter(F.col("drift_sec") <= F.lit(MAX_FUTURE_DRIFT))
  .drop("drift_sec")
  # Только завершённые игры match3
  .filter(F.col("event_type") == "m3.level_finished")
  .filter(F.col("reason").isin("completed", "failed"))
  # Отсекаем лигу чемпионов (в MyM это chain вида WL_*)
  .filter(~F.col("chain").like("WL_%"))
  .filter(F.col("client_time") > DATE_HARD_CUTOFF)
  .filter(F.col("partition_date") > DATE_HARD_CUTOFF)
  # Оставляем только релевантных пользователей
  .join(F.broadcast(relevant_users), on="user_id", how="inner")
  # Дополнительные поля для анализа
  .withColumn("failed", F.when(F.col("reason") == "failed", 1).otherwise(0))
  .withColumn(
    "balance_id",
    F.expr("CAST(regexp_extract(get_json_object(event_payload, '$.balance_id'), 'st0*([0-9]+)', 1) AS INT)"),
  )
  .select(
    *KEY_USER,
    "traffic_type",
    "payer_type",
    "failed",
    "ab_group",
    F.col("attempt").cast("int").alias("attempt"),
    F.json_tuple("user_payload", "tech.platform_name").alias("platform_name"),
  )
  .withColumn("first_attempt", F.when(F.col("attempt") == 1, 1).otherwise(0))
)

# Прицепляем исходы отдельным джойном и восстанавливаем reason_seg из флагов (для раскраски точек)
objects = (
  objects
  .join(outcomes, on=outcome_keys, how="left")
  .fillna(0, subset=OUTCOME_FLAGS)
  .withColumn(
    "reason_seg",
    F.when(F.col("FW") == 1, "far_win")
     .when(F.col("CW") == 1, "close_win")
     .when(F.col("CF") == 1, "close_fail")
     .when(F.col("FF") == 1, "far_fail")
     .otherwise("unknown"),
  )
  .withColumn("level_cohort", F.floor(F.col("balance_id") / F.lit(LEVEL_COHORT_FRACTION)))
)

# Пишем raw objects в Unity Catalog table, которую читает Databricks App
(
  objects.write
  .mode("overwrite")
  .option("mergeSchema", "false")
  .option("overwriteSchema", "true")
  .partitionBy("partition_date")
  .saveAsTable(raw_objects_table)
)

# Графики строим только на ограниченном (тестовом) датасете
if RUN_ON_TEST_USERS:
  df = spark.table(raw_objects_table).toPandas()
  display(df)
  print(f"Размер датасета: {len(df)}")
  for user in TEST_USERS:
    display(px.scatter(df[df.user_id == user], x="client_time", y="balance_id",
                       color="reason_seg", color_discrete_map=COLOR_MAP))